# 10 — Fußdomäne: Positionskurve und Lower-Limb-Modell

Beantwortet die Kernfrage für die Einlegesohle: **Wie gut überträgt
sich ein am Rumpf trainiertes Modell auf das Bein bzw. den Fuß, und
hilft Training mit beinnahen Daten?**

**Teil A — Positionskurve (Zero-Shot).** Das Basismodell (SisFall)
wird auf denselben Stürzen und Alltagsaktivitäten an neun
Körperpositionen bewertet (CAPPIMU, synchron aufgezeichnet), ergänzt
um Fußdaten ohne Stürze (ImbalanSense, Smart-Insole).

**Teil B — Lower-Limb-Training (5-fach, subjektunabhängig).**

| Variante | Training |
|---|---|
| A | Basismodell (SisFall), unverändert |
| K | A + Fine-Tuning auf KFall |
| B1 | K + Fine-Tuning auf CAPPIMU-Knie |
| B2 | K + Fine-Tuning auf CAPPIMU-Knie **+ ImbalanSense-Fußdaten** |

Smart-Insole wird **nie trainiert**, sondern nur als externer
Fehlalarmtest verwendet (Lizenz: nur nicht-kommerziell).

**Datenlage und Einschränkungen**

| Datensatz | Position | Stürze | Lizenz |
|---|---|---|---|
| CAPPIMU | 9 Positionen inkl. Knie (oberer Schienbeinbereich) | ja, ~20/Proband | nicht angegeben |
| ImbalanSense | Fuß (Moticon) | nein | CC-BY 4.0 |
| Smart-Insole v1.0 | Fuß (Moticon) | nein | nicht-kommerziell |

CAPPIMU beschriftet die gesamte Sturzsitzung als `fall`. Einzelne
Aufpraelle werden auf der Brustposition detektiert und über die
gemeinsamen Zeitstempel auf alle Positionen übertragen
(`fl.load_cappimu`). Diese Beschriftung ist eine Näherung.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")
assert hasattr(fl, "load_cappimu"), "fall_lib.py auf GitHub ist noch die alte Version - bitte pushen."

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)

def hole(name, ziel):
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        vorhanden = "\n    ".join(p.name for p in sorted(Path(BASE).iterdir()))
        raise FileNotFoundError(f"'{name}' nicht gefunden unter {BASE}\n  Vorhanden:\n    {vorhanden}")
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn()
    pickle.dump(r, open(p, "wb"))
    return r

from tensorflow import keras
basis = keras.models.load_model(MODEL, compile=False)
basis.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

run = fl.new_run("10_fussdomaene")
ergebnisse = {}
print("Konfiguration:", fl.default_config())

## Datenpfade

CAPPIMU wird **direkt von Drive** gelesen: wenige große Dateien
(~30 MB je Position und Proband), das Kopieren aller ~20 GB lohnt
nicht. Gelesen werden nur die benötigten Positionen.

In [ ]:
# CAPPIMU: Ordner, der subject_1 ... subject_30 enthaelt
CAP_ROOT = next(Path(f"{BASE}/CAPPIMU Dataset").rglob("subject_1")).parent
print("CAPPIMU:", CAP_ROOT)

# ImbalanSense: entpackte .txt oder raw_recordings.zip
IMB = Path("/content/ImbalanSense")
if not IMB.exists():
    src = Path(f"{BASE}/ImbalanSense")
    if any(src.rglob("*.txt")):
        hole("ImbalanSense", str(IMB))
    else:
        z = next(src.rglob("raw_recordings*.zip"))
        zipfile.ZipFile(z).extractall(IMB)
print("ImbalanSense:", sum(1 for _ in IMB.rglob("*.txt")), "Dateien")

# Smart-Insole: nur Annotated_Data
SI = Path("/content/SmartInsole")
if not SI.exists():
    src = next(Path(f"{BASE}/SmartInsole_Dataset_v1_0_Public").rglob("Annotated_Data"))
    shutil.copytree(src, SI)
print("Smart-Insole:", sum(1 for _ in SI.rglob("*.xlsx")), "Dateien")

## Teil A — Positionskurve (Zero-Shot)

Kennzahlen je Position:
* **Se** auf aufprallzentrierten Sturzfenstern
* **FP/Std** auf allen ADL-Fenstern (Schrittweite 1 s, Schwelle 0,5)
* **Peak Sturz** (Median) und **Peak ADL** (99. Perzentil) von |a| in g —
  je kleiner der Abstand, desto schwerer die Trennung

In [ ]:
def fenster(trials):
    X, y, g, a = fl.build_windows(trials, mode="standard")
    peaks = fl.acc_magnitude(X).max(axis=1)        # vor Normalisierung
    return fl.znorm(X), y, g, a, peaks

def fp_pro_stunde(n_fp, n_adl):
    h = n_adl * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return round(n_fp / h, 2) if h else None

def bewerte(model, X, y, peaks, name, thr=0.5):
    r = fl.evaluate(model, X, y, name, threshold=thr)
    r["fp_per_hour"] = fp_pro_stunde(r["FP"], int((y == 0).sum()))
    r["n_falls"] = int(y.sum()); r["n_adl"] = int((y == 0).sum())
    r["peak_fall_median_g"] = round(float(np.median(peaks[y == 1])), 2) if y.sum() else None
    r["peak_adl_p99_g"] = round(float(np.percentile(peaks[y == 0], 99)), 2)
    return r

kurve = {}
for pos in fl.CAPPIMU_POSITIONS:
    t = time.time()
    tr = cached(f"cappimu_{pos}", lambda: fl.load_cappimu(CAP_ROOT, pos))
    if not tr:
        print(f"[!] {pos}: keine Daten"); continue
    X, y, g, a, pk = fenster(tr)
    kurve[f"CAPPIMU {pos}"] = bewerte(basis, X, y, pk, f"CAPPIMU {pos}")
    print(f"     FP/Std={kurve[f'CAPPIMU {pos}']['fp_per_hour']}  ({time.time()-t:.0f}s)")
    del tr, X

In [ ]:
imb = cached("imbalansense", lambda: fl.load_imbalansense(IMB))
si  = cached("smart_insole", lambda: fl.load_smart_insole(SI))

X_imb, y_imb, g_imb, a_imb, pk_imb = fenster(imb)
X_si,  y_si,  g_si,  a_si,  pk_si  = fenster(si)
grp_si = np.array([s.split("_")[1].rstrip("0123456789") for s in g_si])

kurve["Fuss ImbalanSense"] = bewerte(basis, X_imb, y_imb, pk_imb, "Fuss ImbalanSense")
kurve["Fuss Smart-Insole"] = bewerte(basis, X_si, y_si, pk_si, "Fuss Smart-Insole")

print("\nImbalanSense nach Bedingung:")
for c in ["norm", "vr", "shaking"]:
    m = np.array([s.startswith(c) for s in a_imb])
    if m.any(): kurve[f"Fuss IMB {c}"] = bewerte(basis, X_imb[m], y_imb[m], pk_imb[m], f"  {c}")
print("\nSmart-Insole nach Gruppe:")
for gr in ["S", "EL", "PD"]:
    m = grp_si == gr
    if m.any(): kurve[f"Fuss SI {gr}"] = bewerte(basis, X_si[m], y_si[m], pk_si[m], f"  {gr}")
ergebnisse["positionskurve"] = kurve

In [ ]:
import matplotlib.pyplot as plt

reihen = [p for p in ["head", "chest", "arm_l", "arm_r", "wrist_l", "wrist_r",
                      "pocket", "knee_l", "knee_r"] if f"CAPPIMU {p}" in kurve]
namen  = [f"CAPPIMU {p}" for p in reihen] + ["Fuss ImbalanSense", "Fuss Smart-Insole"]
labels = reihen + ["Fuß (IMB)", "Fuß (SI)"]

print(f"{'Position':<22}{'Se':>7}{'Sp':>7}{'FP/Std':>9}{'Peak Sturz':>12}{'Peak ADL p99':>14}")
print("-" * 71)
for n in namen:
    r = kurve[n]
    se = f"{r['Se']:.3f}" if r["n_falls"] else "  -  "
    pf = f"{r['peak_fall_median_g']:.2f}" if r["peak_fall_median_g"] else "-"
    print(f"{n:<22}{se:>7}{r['Sp']:>7.3f}{r['fp_per_hour']:>9.1f}{pf:>12}{r['peak_adl_p99_g']:>14.2f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(namen))
se = [kurve[n]["Se"] if kurve[n]["n_falls"] else np.nan for n in namen]
fp = [kurve[n]["fp_per_hour"] for n in namen]
ax[0].plot(x, se, "o-", color="#80BA24", label="Se")
ax0b = ax[0].twinx(); ax0b.bar(x, fp, alpha=.35, color="#9C132E", label="FP/Std")
ax[0].set_xticks(x); ax[0].set_xticklabels(labels, rotation=35, ha="right")
ax[0].set_ylabel("Sensitivität"); ax0b.set_ylabel("Fehlalarme / Stunde")
ax[0].set_title("Basismodell (SisFall) je Position")

pf = [kurve[n]["peak_fall_median_g"] or np.nan for n in namen]
pa = [kurve[n]["peak_adl_p99_g"] for n in namen]
ax[1].plot(x, pf, "o-", color="#80BA24", label="Sturz (Median)")
ax[1].plot(x, pa, "s--", color="#4A5C66", label="ADL (99. Perzentil)")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels, rotation=35, ha="right")
ax[1].set_ylabel("max |a| je Fenster [g]"); ax[1].legend()
ax[1].set_title("Abstand Sturz – Alltag")
plt.tight_layout()
plt.savefig(run / "positionskurve.png", dpi=200, bbox_inches="tight")
plt.show()

## Teil B — Lower-Limb-Training

Fünffache, subjektunabhängige Kreuzvalidierung, getrennt für CAPPIMU
(Probanden) und ImbalanSense (Probanden). Falte *i* testet jeweils auf
den Probanden beider Datensätze, die nicht im Training waren.
Smart-Insole ist in keiner Falte im Training.

ADL-Fenster werden **für das Training** gekappt (Rechenzeit); bewertet
wird immer auf allen Testfenstern.

In [ ]:
N_FOLDS   = 5
MAX_ADL_CAP = 20000      # ADL-Trainingsfenster je Falte aus CAPPIMU
MAX_ADL_IMB = 10000      # ADL-Trainingsfenster je Falte aus ImbalanSense
rng = np.random.default_rng(42)

def kappe(idx, y, max_adl):
    f = idx[y[idx] == 1]; a = idx[y[idx] == 0]
    if len(a) > max_adl:
        a = rng.choice(a, max_adl, replace=False)
    return np.r_[f, a]

# CAPPIMU Knie (beide Beine, gleicher Proband -> gleiche Gruppe)
knie = cached("cappimu_knee_l", lambda: fl.load_cappimu(CAP_ROOT, "knee_l")) + \
       cached("cappimu_knee_r", lambda: fl.load_cappimu(CAP_ROOT, "knee_r"))
X_c, y_c, g_c, a_c, pk_c = fenster(knie)
print(f"CAPPIMU Knie: {len(y_c)} Fenster, {int(y_c.sum())} Stuerze, {len(np.unique(g_c))} Probanden")
print(f"ImbalanSense: {len(y_imb)} Fenster, {len(np.unique(g_imb))} Probanden")
print(f"Smart-Insole: {len(y_si)} Fenster (nur Test)")

# KFall-Zwischenstufe (wie Notebook 07)
kf = fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
                   label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"))
X_kf, y_kf, _, _ = fl.build_windows(kf, mode="standard")
X_kf = fl.znorm(X_kf)
t = time.time(); m_kfall = fl.finetune(basis, X_kf, y_kf)
print(f"KFall-Fine-Tuning: {time.time()-t:.0f}s")
del X_kf

In [ ]:
sp_cap = fl.group_kfold_splits(g_c, y_c, n_splits=N_FOLDS)
sp_imb = fl.group_kfold_splits(g_imb, None, n_splits=N_FOLDS)

def alle_tests(m, te_c, te_i, tag):
    return {
        "knie": fl.evaluate(m, X_c[te_c], y_c[te_c], f"{tag} Knie"),
        "imb":  fl.evaluate(m, X_imb[te_i], y_imb[te_i], f"{tag} Fuss IMB"),
        "si":   fl.evaluate(m, X_si, y_si, f"{tag} Fuss SI"),
    }

falten = {k: [] for k in ["A", "K", "B1", "B2"]}
for i, ((tr_c, te_c), (tr_i, te_i)) in enumerate(zip(sp_cap, sp_imb), 1):
    print(f"\nFalte {i}/{N_FOLDS}: Knie-Test {len(te_c)} Fenster "
          f"({int(y_c[te_c].sum())} Stuerze), IMB-Test {len(te_i)} Fenster")
    t = time.time()
    falten["A"].append(alle_tests(basis,   te_c, te_i, "  A "))
    falten["K"].append(alle_tests(m_kfall, te_c, te_i, "  K "))

    i_c = kappe(tr_c, y_c, MAX_ADL_CAP)
    m_b1 = fl.finetune(m_kfall, X_c[i_c], y_c[i_c])
    falten["B1"].append(alle_tests(m_b1, te_c, te_i, "  B1"))

    i_i = rng.choice(tr_i, min(MAX_ADL_IMB, len(tr_i)), replace=False)
    X_tr = np.concatenate([X_c[i_c], X_imb[i_i]])
    y_tr = np.concatenate([y_c[i_c], y_imb[i_i]])
    m_b2 = fl.finetune(m_kfall, X_tr, y_tr)
    falten["B2"].append(alle_tests(m_b2, te_c, te_i, "  B2"))
    print(f"  ({time.time()-t:.0f}s)")
    del m_b1, m_b2

## Zusammengefasstes Ergebnis

In [ ]:
h = lambda n: n * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
zusammen = {}
print(f"{'Var.':<5}{'Knie Se':>9}{'95%-KI':>18}{'Knie Sp':>9}{'IMB FP/Std':>12}{'SI FP/Std':>11}")
print("-" * 64)
for k, fl_ in falten.items():
    knie_p = fl.pool_folds([f["knie"] for f in fl_])
    imb_fp = sum(f["imb"]["FP"] for f in fl_); imb_n = sum(f["imb"]["FP"] + f["imb"]["TN"] for f in fl_)
    si_fp  = np.mean([f["si"]["FP"] for f in fl_]); si_n = len(y_si)
    zusammen[k] = {"knie": knie_p,
                   "imb_fp_per_hour": round(imb_fp / h(imb_n), 2),
                   "si_fp_per_hour": round(si_fp / h(si_n), 2)}
    ci = knie_p["Se_ci95"]
    print(f"{k:<5}{knie_p['Se']:>9.3f}   [{ci[0]:.3f}; {ci[1]:.3f}]{knie_p['Sp']:>9.3f}"
          f"{zusammen[k]['imb_fp_per_hour']:>12.1f}{zusammen[k]['si_fp_per_hour']:>11.1f}")
ergebnisse["lower_limb"] = {"zusammen": zusammen, "falten": falten}

## Gepaarte Vergleiche

* **B1 vs K:** Bringt Knie-Training etwas gegenüber dem Rumpfmodell?
* **B2 vs B1:** Helfen zusätzliche Fußdaten (ImbalanSense) — oder schaden
  sie der Sensitivität wie das Hard-Negative-Training in Notebook 08?

In [ ]:
vgl = {}
for a, b, met, quelle in [("K", "B1", "Se", "knie"), ("K", "B1", "Sp", "knie"),
                          ("B1", "B2", "Se", "knie"), ("B1", "B2", "Sp", "knie"),
                          ("B1", "B2", "Sp", "imb"),  ("B1", "B2", "Sp", "si")]:
    print(f"\n{b} vs {a} — {quelle}")
    vgl[f"{b}_vs_{a}_{met}_{quelle}"] = fl.paired_comparison(
        [f[quelle] for f in falten[a]], [f[quelle] for f in falten[b]], a, b, metric=met)
ergebnisse["vergleiche"] = vgl

## Endmodell für den Pilotversuch

B2 auf **allen** CAPPIMU-Knie- und ImbalanSense-Daten trainiert.
Wird in Notebook 09 als zweites Modell neben dem Basismodell auf die
eigenen ESP32-Aufnahmen angewendet.

In [ ]:
i_c = kappe(np.arange(len(y_c)), y_c, MAX_ADL_CAP)
i_i = rng.choice(np.arange(len(y_imb)), min(MAX_ADL_IMB, len(y_imb)), replace=False)
m_final = fl.finetune(m_kfall, np.concatenate([X_c[i_c], X_imb[i_i]]),
                      np.concatenate([y_c[i_c], y_imb[i_i]]))
pfad = f"{BASE}/results/fall_model_lowerlimb.keras"
m_final.save(pfad)
print("gesichert:", pfad)

fl.save_results(run, ergebnisse,
    config={**fl.default_config(), "n_folds": N_FOLDS,
            "max_adl_cap": MAX_ADL_CAP, "max_adl_imb": MAX_ADL_IMB,
            "cappimu_ref": "chest", "cappimu_min_g": 2.5},
    notes=("Positionskurve (CAPPIMU 9 Positionen + Fuss) und "
           "Lower-Limb-Training K/B1/B2 mit 5-facher CV. "
           "Smart-Insole nur externer Test."))